# Comparação entre `visual-memory/ConvAI2` e `visual-memory/PersonaChat`

Este notebook faz uma análise comparativa completa entre os dois datasets publicados no namespace `visual-memory/`:

- **`visual-memory/ConvAI2`** (config `both_revised_no_candidates`) — uma conversa por linha, com `your_persona`/`partner_persona` como `list[string]` e `turns` (`turn_id`/`text`/`response`). Deriva da variante *Both Revised* do ParlAI, sem candidatos.
- **`visual-memory/PersonaChat`** — colunas `dialog_id`, `your_persona_original`/`partner_persona_original` (JSON), `your_persona_revised`/`partner_persona_revised` (JSON), `dialog` (JSON de turns com `text`/`response`/`candidates`) e 4 colunas `*_enhanced_persona_*`.

**Perguntas a responder:** um é complemento do outro? Tudo que existe em um existe no outro? O que cada um adiciona? A hipótese central é que o ConvAI2 usa personas **revisadas**, então `ConvAI2.your_persona` deve corresponder a `PersonaChat.your_persona_revised`, e o PersonaChat seria um subconjunto canônico reanotado de um ConvAI2 maior.

Dependências: `datasets` e `pandas` (disponíveis no `.venv` do projeto). O notebook é somente leitura — não publica nem muta datasets.

## 1. Configuração e carregamento

In [1]:
from __future__ import annotations

import hashlib
import json
from typing import Any, Iterable, Sequence

import pandas as pd
from datasets import DatasetDict, load_dataset

pd.set_option("display.max_colwidth", 120)

CONVAI2_REPO = "visual-memory/ConvAI2"
PERSONACHAT_REPO = "visual-memory/PersonaChat"

convai2 = load_dataset(CONVAI2_REPO)
personachat = load_dataset(PERSONACHAT_REPO)

print("ConvAI2:")
print(convai2)
print("\nPersonaChat:")
print(personachat)

ConvAI2:
DatasetDict({
    train: Dataset({
        features: ['conversation_id', 'your_persona', 'partner_persona', 'turns', 'source_split'],
        num_rows: 17878
    })
    validation: Dataset({
        features: ['conversation_id', 'your_persona', 'partner_persona', 'turns', 'source_split'],
        num_rows: 1000
    })
})

PersonaChat:
DatasetDict({
    train: Dataset({
        features: ['dialog_id', 'your_persona_original', 'partner_persona_original', 'your_persona_revised', 'partner_persona_revised', 'dialog', 'your_enhanced_persona_original', 'partner_enhanced_persona_original', 'your_enhanced_persona_revised', 'partner_enhanced_persona_revised'],
        num_rows: 8939
    })
    validation: Dataset({
        features: ['dialog_id', 'your_persona_original', 'partner_persona_original', 'your_persona_revised', 'partner_persona_revised', 'dialog', 'your_enhanced_persona_original', 'partner_enhanced_persona_original', 'your_enhanced_persona_revised', 'partner_enhanced_persona_

## 2. Comparação de esquemas

Tabela lado a lado de colunas, tipos e descrição. Os datasets divergem no formato da persona (lista nativa vs JSON-string) e nos metadados de anotação (PersonaChat adiciona personas originais, *enhanced* e candidatos).

In [2]:
def feature_table(ds: DatasetDict) -> pd.DataFrame:
    sample_split = next(iter(ds))
    feats = ds[sample_split].features
    return pd.DataFrame(
        {
            "coluna": list(feats.keys()),
            "tipo": [str(feats[c]) for c in feats],
        }
    )

schema_rows = []
for name, ds in [("ConvAI2", convai2), ("PersonaChat", personachat)]:
    for _, row in feature_table(ds).iterrows():
        schema_rows.append({"dataset": name, "coluna": row["coluna"], "tipo": row["tipo"]})

pd.DataFrame(schema_rows)

,dataset,coluna,tipo
0,ConvAI2,conversation_id,Value('string')
1,ConvAI2,your_persona,List(Value('string'))
2,ConvAI2,partner_persona,List(Value('string'))
3,ConvAI2,turns,"List({'turn_id': Value('int64'), 'text': Value('string'), 'response': Value('string')})"
4,ConvAI2,source_split,Value('string')
5,PersonaChat,dialog_id,Value('large_string')
6,PersonaChat,your_persona_original,Value('large_string')
7,PersonaChat,partner_persona_original,Value('large_string')
8,PersonaChat,your_persona_revised,Value('large_string')
9,PersonaChat,partner_persona_revised,Value('large_string')


## 3. Comparação de splits e tamanhos

Ambos têm validação com 1.000 conversas. O ConvAI2 tem muito mais treino (17.878 vs 8.939); o PersonaChat adiciona um split de teste (968) ausente no ConvAI2.

In [3]:
def sizes_table(ds: DatasetDict, name: str) -> pd.DataFrame:
    return pd.DataFrame(
        [{"dataset": name, "split": split, "conversas": len(ds[split])} for split in ds]
    )

sizes = pd.concat([sizes_table(convai2, "ConvAI2"), sizes_table(personachat, "PersonaChat")], ignore_index=True)
display(sizes)

totals = sizes.groupby("dataset")["conversas"].sum().reset_index().rename(columns={"conversas": "total"})
display(totals)

,dataset,split,conversas
0,ConvAI2,train,17878
1,ConvAI2,validation,1000
2,PersonaChat,train,8939
3,PersonaChat,validation,1000
4,PersonaChat,test,968


,dataset,total
0,ConvAI2,18878
1,PersonaChat,10907


## 4. Normalização de personas e chave canônica

O ConvAI2 armazena personas como `list[string]`; o PersonaChat armazena como string JSON. `normalize_persona` aceita os dois formatos: faz `json.loads` quando necessário, aplica strip em cada frase e descarta frases vazias. A chave da persona é ordenação-independente (`tuple(sorted(...))`) para tolerar diferenças de ordem entre as anotações.

In [4]:
def normalize_persona(value: Any) -> tuple[str, ...]:
    """Converte persona (list[str] ou str JSON) em tupla ordenada de frases limpas."""
    if value is None:
        return ()
    if isinstance(value, str):
        value = value.strip()
        if not value or value == "[]":
            return ()
        try:
            value = json.loads(value)
        except json.JSONDecodeError:
            value = [value]
    if isinstance(value, str):
        value = [value]
    phrases = [str(p).strip() for p in value]
    phrases = [p for p in phrases if p]
    return tuple(sorted(phrases))


def persona_key(value: Any) -> str:
    """Hash determinístico e ordem-independente da persona."""
    phrases = normalize_persona(value)
    return hashlib.sha256("\x1f".join(phrases).encode("utf-8")).hexdigest()


# Sanity check: amostras dos dois formatos
print("ConvAI2 your_persona[0]:", convai2["train"][0]["your_persona"])
print("-> normalizado:", normalize_persona(convai2["train"][0]["your_persona"]))
print()
print("PersonaChat your_persona_revised[0]:", personachat["train"][0]["your_persona_revised"])
print("-> normalizado:", normalize_persona(personachat["train"][0]["your_persona_revised"]))

ConvAI2 your_persona[0]: ['i love to redesign houses.', 'killing for sport is my hobby.', 'i shot an arrow the other day !.', 'i like to get dressed up.']
-> normalizado: ('i like to get dressed up.', 'i love to redesign houses.', 'i shot an arrow the other day !.', 'killing for sport is my hobby.')

PersonaChat your_persona_revised[0]: ["i love to redesign houses.","killing for sport is my hobby.","i shot an arrow the other day !.","i like to get dressed up."]
-> normalizado: ('i like to get dressed up.', 'i love to redesign houses.', 'i shot an arrow the other day !.', 'killing for sport is my hobby.')


In [5]:
# Verifica se alguma persona fica vazia após a normalização em qualquer coluna relevante.

def collect_persona_values(ds: DatasetDict, columns: Iterable[str]) -> list[tuple[str, str, int, Any]]:
    """Retorna (split, coluna, indice, valor-bruto) para inspeção."""
    out = []
    for split in ds:
        for col in columns:
            if col not in ds[split].column_names:
                continue
            for i, val in enumerate(ds[split][col]):
                out.append((split, col, i, val))
    return out

convai2_persona_cols = ["your_persona", "partner_persona"]
pc_persona_cols = [
    "your_persona_original", "partner_persona_original",
    "your_persona_revised", "partner_persona_revised",
]

empties = []
for split, col, i, val in collect_persona_values(convai2, convai2_persona_cols):
    if not normalize_persona(val):
        empties.append({"dataset": "ConvAI2", "split": split, "coluna": col, "indice": i})
for split, col, i, val in collect_persona_values(personachat, pc_persona_cols):
    if not normalize_persona(val):
        empties.append({"dataset": "PersonaChat", "split": split, "coluna": col, "indice": i})

print(f"Personas vazias após normalização: {len(empties)}")
if empties:
    display(pd.DataFrame(empties))

Personas vazias após normalização: 0


## 5. Conjuntos de personas únicas

Para o PersonaChat separamos três conjuntos — `pc_original`, `pc_revised` e `pc_enhanced` — porque a hipótese depende de saber a **qual** versão do PersonaChat as personas do ConvAI2 correspondem. Para o ConvAI2 há um único conjunto.

In [6]:
def persona_set(ds: DatasetDict, columns: Iterable[str]) -> set[str]:
    cols = [c for c in columns if c in next(iter(ds.values())).column_names]
    out: set[str] = set()
    for split in ds:
        if not cols:
            continue
        data = ds[split][:]
        for col in cols:
            for val in data[col]:
                key = persona_key(val)
                if key:
                    out.add(key)
    return out

convai2_personas = persona_set(convai2, convai2_persona_cols)
pc_original = persona_set(personachat, ["your_persona_original", "partner_persona_original"])
pc_revised = persona_set(personachat, ["your_persona_revised", "partner_persona_revised"])
pc_enhanced = persona_set(
    personachat,
    ["your_enhanced_persona_original", "partner_enhanced_persona_original",
     "your_enhanced_persona_revised", "partner_enhanced_persona_revised"],
)

persona_counts = pd.DataFrame(
    {
        "conjunto": ["ConvAI2 (todas)", "PersonaChat original", "PersonaChat revised", "PersonaChat enhanced"],
        "personas_unicas": [len(convai2_personas), len(pc_original), len(pc_revised), len(pc_enhanced)],
    }
)
persona_counts

,conjunto,personas_unicas
0,ConvAI2 (todas),10321
1,PersonaChat original,7027
2,PersonaChat revised,6162
3,PersonaChat enhanced,13189


## 6. Sobreposição de personas

Teste da hipótese: a interseção `ConvAI2 ∩ PersonaChat_revised` deve ser muito maior que `ConvAI2 ∩ PersonaChat_original`, confirmando que o ConvAI2 usa a versão revisada das personas. Calculamos interseção, diferenças e índice de Jaccard.

In [7]:
def overlap_table(a: set, b: set, label_a: str, label_b: str) -> dict:
    inter = a & b
    union = a | b
    return {
        "comparacao": f"{label_a} ∩ {label_b}",
        "tamanho_a": len(a),
        "tamanho_b": len(b),
        "intersecao": len(inter),
        "so_a": len(a - b),
        "so_b": len(b - a),
        "jaccard": round(len(inter) / len(union), 4) if union else 0.0,
    }

overlap_rows = [
    overlap_table(convai2_personas, pc_original, "ConvAI2", "PC original"),
    overlap_table(convai2_personas, pc_revised, "ConvAI2", "PC revised"),
    overlap_table(convai2_personas, pc_enhanced, "ConvAI2", "PC enhanced"),
    overlap_table(pc_original, pc_revised, "PC original", "PC revised"),
]
pd.DataFrame(overlap_rows)

,comparacao,tamanho_a,tamanho_b,intersecao,so_a,so_b,jaccard
0,ConvAI2 ∩ PC original,10321,7027,0,10321,7027,0.0000
1,ConvAI2 ∩ PC revised,10321,6162,3787,6534,2375,0.2983
2,ConvAI2 ∩ PC enhanced,10321,13189,0,10321,13189,0.0000
3,PC original ∩ PC revised,7027,6162,0,7027,6162,0.0000


In [8]:
best_rev = overlap_table(convai2_personas, pc_revised, "ConvAI2", "PC revised")
best_orig = overlap_table(convai2_personas, pc_original, "ConvAI2", "PC original")
if best_rev["intersecao"] > best_orig["intersecao"]:
    print(f"Hipótese confirmada: personas do ConvAI2 correspondem à versão REVISED do PersonaChat.")
    print(f"  ConvAI2 ∩ PC_revised = {best_rev['intersecao']} ({best_rev['jaccard']} de Jaccard)")
    print(f"  ConvAI2 ∩ PC_original = {best_orig['intersecao']} ({best_orig['jaccard']} de Jaccard)")
else:
    print(f"Hipótese refutada: personas do ConvAI2 NÃO correspondem melhor à versão revised.")
    print(f"  ConvAI2 ∩ PC_revised = {best_rev['intersecao']} ({best_rev['jaccard']} de Jaccard)")
    print(f"  ConvAI2 ∩ PC_original = {best_orig['intersecao']} ({best_orig['jaccard']} de Jaccard)")

print()
print(f"Personas só no ConvAI2 (não estão em nenhuma versão do PersonaChat): "
      f"{len(convai2_personas - pc_revised - pc_original)}")
print(f"Personas só no PersonaChat revised (não no ConvAI2): {len(pc_revised - convai2_personas)}")

Hipótese confirmada: personas do ConvAI2 correspondem à versão REVISED do PersonaChat.
  ConvAI2 ∩ PC_revised = 3787 (0.2983 de Jaccard)
  ConvAI2 ∩ PC_original = 0 (0.0 de Jaccard)

Personas só no ConvAI2 (não estão em nenhuma versão do PersonaChat): 6534
Personas só no PersonaChat revised (não no ConvAI2): 2375


## 7. Sobreposição de diálogos/conversas (igualdade bruta)

Chave de conversa = `hashlib.sha256` da sequência canonicalizada de pares `(text, response)` normalizados (lower + strip). Isso ignora metadados (turn_id, reward, candidates) e foca no conteúdo do diálogo. **Atenção:** esta métrica é *exata e bruta* — ela não tolera a diferença de normalização de contrações (`i'm` vs `i am`) que existe entre os datasets, então ela **subestima** a sobreposição real. A seção 7b corrige isso com expansão de contrações e similaridade fuzzy.

In [9]:
import re

def _norm_text(s: Any) -> str:
    return " ".join(str(s).lower().strip().split())


def dialog_key_personachat(dialog: Any) -> str | None:
    # PersonaChat: coluna `dialog` como str JSON de turns com `text`/`response` (igualdade EXATA, bruta).
    if isinstance(dialog, str):
        try:
            dialog = json.loads(dialog)
        except json.JSONDecodeError:
            return None
    if not isinstance(dialog, (list, tuple)) or not dialog:
        return None
    seq = [f"{_norm_text(turn.get('text', ''))}\x1f{_norm_text(turn.get('response', ''))}" for turn in dialog]
    return hashlib.sha256("\x1e".join(seq).encode("utf-8")).hexdigest()


def dialog_key_convai2(turns: Any) -> str | None:
    # ConvAI2: coluna `turns` como list[struct] com `text`/`response` (igualdade EXATA, bruta).
    if turns is None or not isinstance(turns, (list, tuple)) or not turns:
        return None
    seq = [f"{_norm_text(turn.get('text', ''))}\x1f{_norm_text(turn.get('response', ''))}" for turn in turns]
    return hashlib.sha256("\x1e".join(seq).encode("utf-8")).hexdigest()


# --- Normalização de contrações -------------------------------------------------
# Os dois datasets usam a MESMA conversa, mas o PersonaChat EXPANDE contrações
# (ex.: `i'm` -> `i am`, `don't` -> `do not`, `won't` -> `will not`) enquanto o
# ConvAI2 mantém a forma contraída. Por isso a igualdade exata (bruta)
# subestima fortemente a sobreposição de diálogos (ver seção 7b). A
# canonicalização abaixo expande um conjunto fixo de contrações inglesas para
# permitir comparação justa; casos ambíguos (`i'd` -> `i would` *ou* `i had`)
# ficam a cargo da métrica de similaridade fuzzy (SequenceMatcher).
_CONTRACTIONS: list[tuple[str, str]] = [
    (r"\bi'm\b", "i am"), (r"\bi've\b", "i have"), (r"\bi'll\b", "i will"), (r"\bi'd\b", "i would"),
    (r"\byou're\b", "you are"), (r"\byou've\b", "you have"), (r"\byou'll\b", "you will"), (r"\byou'd\b", "you would"),
    (r"\bhe's\b", "he is"), (r"\bshe's\b", "she is"), (r"\bit's\b", "it is"),
    (r"\bwe're\b", "we are"), (r"\bwe've\b", "we have"), (r"\bwe'll\b", "we will"), (r"\bwe'd\b", "we would"),
    (r"\bthey're\b", "they are"), (r"\bthey've\b", "they have"), (r"\bthey'll\b", "they will"), (r"\bthey'd\b", "they would"),
    (r"\bthat's\b", "that is"), (r"\bthere's\b", "there is"), (r"\bhere's\b", "here is"),
    (r"\bwhat's\b", "what is"), (r"\bwho's\b", "who is"), (r"\bhow's\b", "how is"),
    (r"\bdon't\b", "do not"), (r"\bdoesn't\b", "does not"), (r"\bdidn't\b", "did not"),
    (r"\bcan't\b", "cannot"), (r"\bwon't\b", "will not"), (r"\bwouldn't\b", "would not"),
    (r"\bshouldn't\b", "should not"), (r"\bcouldn't\b", "could not"),
    (r"\bisn't\b", "is not"), (r"\baren't\b", "are not"), (r"\bwasn't\b", "was not"), (r"\bweren't\b", "were not"),
    (r"\bhasn't\b", "has not"), (r"\bhaven't\b", "have not"), (r"\bhadn't\b", "had not"),
    (r"\bwould've\b", "would have"), (r"\bcould've\b", "could have"), (r"\bshould've\b", "should have"),
    (r"\blet's\b", "let us"),
]
_CONTRACTION_RE = re.compile("|".join(p for p, _ in _CONTRACTIONS))


def expand_text(s: Any) -> str:
    # Lowercase, troca apóstrofo curvo por reto, expande contrações, colapsa espaços.
    s = str(s).lower().replace("’", "'")
    def _rep(m: "re.Match[str]") -> str:
        for pat, repl in _CONTRACTIONS:
            if re.match(pat, m.group(0)):
                return repl
        return m.group(0)
    return " ".join(_CONTRACTION_RE.sub(_rep, s).split())


def _conv_str_pc(dialog: Any) -> str:
    if isinstance(dialog, str):
        try:
            dialog = json.loads(dialog)
        except json.JSONDecodeError:
            return ""
    if not isinstance(dialog, (list, tuple)):
        return ""
    return " \x1e ".join(f"{expand_text(t.get('text', ''))} \x1f {expand_text(t.get('response', ''))}" for t in dialog)


def _conv_str_cv(turns: Any) -> str:
    if not isinstance(turns, (list, tuple)):
        return ""
    return " \x1e ".join(f"{expand_text(t.get('text', ''))} \x1f {expand_text(t.get('response', ''))}" for t in turns)


def dialog_key_personachat_expanded(dialog: Any) -> str | None:
    s = _conv_str_pc(dialog)
    return hashlib.sha256(s.encode("utf-8")).hexdigest() if s else None


def dialog_key_convai2_expanded(turns: Any) -> str | None:
    s = _conv_str_cv(turns)
    return hashlib.sha256(s.encode("utf-8")).hexdigest() if s else None


def conv_similarity(dialog_pc: Any, turns_cv: Any) -> float:
    # Similaridade fuzzy [0,1] entre duas conversas (SequenceMatcher sobre texto canônico).
    import difflib
    return difflib.SequenceMatcher(None, _conv_str_pc(dialog_pc), _conv_str_cv(turns_cv)).ratio()


# Sanity check: chaves brutas diferem; chaves expandidas coincidem; similaridade ~1.0
print("PersonaChat dialog[0] key (bruto):", dialog_key_personachat(personachat["train"][0]["dialog"]))
print("ConvAI2 turns[0]   key (bruto):", dialog_key_convai2(convai2["train"][0]["turns"]))
print("PersonaChat dialog[0] key (exp.):", dialog_key_personachat_expanded(personachat["train"][0]["dialog"]))
print("ConvAI2 turns[0]   key (exp.):", dialog_key_convai2_expanded(convai2["train"][0]["turns"]))
print("Similaridade (exemplo 0):", round(conv_similarity(personachat["train"][0]["dialog"], convai2["train"][0]["turns"]), 4))

PersonaChat dialog[0] key (bruto): c34074b0547f084607fcbf015e6cf39c9fde0efd12070bda40092acc100a918b
ConvAI2 turns[0]   key (bruto): a5b8a8a468ee4ad6bb5e1a862ca4e9e4b9fa9ab26f485cb0abeeceb1014d497a
PersonaChat dialog[0] key (exp.): 9b1c9267f7533db93908cab030f71db52b6e32272398fc59a696417a82cfe507
ConvAI2 turns[0]   key (exp.): 9b1c9267f7533db93908cab030f71db52b6e32272398fc59a696417a82cfe507
Similaridade (exemplo 0): 1.0


In [10]:
personachat["train"][0]["dialog"]

'[{"turn":1,"text":"hi , how are you doing ? i am getting ready to do some cheetah chasing to stay in shape .","response":"you must be very fast . hunting is one of my favorite hobbies .","reward":"","candidates":["my mom was single with 3 boys , so we never left the projects .","i try to wear all black every day . it makes me feel comfortable .","well nursing stresses you out so i wish luck with sister","yeah just want to pick up nba nfl getting old","i really like celine dion . what about you ?","no . i live near farms .","i wish i had a daughter , i am a boy mom . they are beautiful boys though still lucky","yeah when i get bored i play gone with the wind my favorite movie .","hi how are you ? i am eating dinner with my hubby and 2 kids .","were you married to your high school sweetheart ? i was .","that is great to hear ! are you a competitive rider ?","hi , i am doing ok . i am a banker . how about you ?","i am 5 years old","hi there . how are you today ?","i totally understand ho

In [11]:
convai2["train"][0]["turns"]

[{'turn_id': 0,
  'text': "hi , how are you doing ? i'm getting ready to do some cheetah chasing to stay in shape .",
  'response': 'you must be very fast . hunting is one of my favorite hobbies .'},
 {'turn_id': 1,
  'text': 'i am ! for my hobby i like to do canning or some whittling .',
  'response': 'i also remodel homes when i am not out bow hunting .'},
 {'turn_id': 2,
  'text': "that's neat . when i was in high school i placed 6th in 100m dash !",
  'response': "that's awesome . do you have a favorite season or time of year ?"},
 {'turn_id': 3,
  'text': 'i do not . but i do have a favorite meat since that is all i eat exclusively .',
  'response': 'what is your favorite meat to eat ?'},
 {'turn_id': 4,
  'text': 'i would have to say its prime rib . do you have any favorite foods ?',
  'response': 'i like chicken or macaroni and cheese .'},
 {'turn_id': 5,
  'text': 'do you have anything planned for today ? i think i am going to do some canning .',
  'response': 'i am going to wa

In [12]:
def convai2_dialog_keys(ds: DatasetDict) -> dict[str, set[str]]:
    out: dict[str, set[str]] = {}
    for split in ds:
        cols = ds[split].column_names
        turns_list = ds[split][:]["turns"] if "turns" in cols else [r["turns"] for r in ds[split]]
        out[split] = {k for k in (dialog_key_convai2(t) for t in turns_list) if k}
    return out

def personachat_dialog_keys(ds: DatasetDict) -> dict[str, set[str]]:
    out: dict[str, set[str]] = {}
    for split in ds:
        cols = ds[split].column_names
        if "dialog" in cols:
            dialogs = ds[split][:]["dialog"]
        else:
            dialogs = [r["dialog"] for r in ds[split]]
        out[split] = {k for k in (dialog_key_personachat(d) for d in dialogs) if k}
    return out

convai2_dialogs = convai2_dialog_keys(convai2)
pc_dialogs = personachat_dialog_keys(personachat)

convai2_all = set().union(*convai2_dialogs.values())
pc_all = set().union(*pc_dialogs.values())

rows = []
for split, keys in pc_dialogs.items():
    in_convai2_any = keys & convai2_all
    rows.append({
        "origem": "PersonaChat",
        "split": split,
        "conversas": len(keys),
        "no_convai2_global": len(in_convai2_any),
        "fora_do_convai2": len(keys - convai2_all),
        "pct_no_convai2": round(len(in_convai2_any) / len(keys) * 100, 1) if keys else 0.0,
    })
for split, keys in convai2_dialogs.items():
    in_pc_any = keys & pc_all
    rows.append({
        "origem": "ConvAI2",
        "split": split,
        "conversas": len(keys),
        "no_personachat_global": len(in_pc_any),
        "fora_do_personachat": len(keys - pc_all),
        "pct_no_personachat": round(len(in_pc_any) / len(keys) * 100, 1) if keys else 0.0,
    })
pd.DataFrame(rows)

,origem,split,conversas,no_convai2_global,fora_do_convai2,pct_no_convai2,no_personachat_global,fora_do_personachat,pct_no_personachat
0,PersonaChat,train,8939,453.0,8486.0,5.1,NaN,NaN,NaN
1,PersonaChat,validation,1000,57.0,943.0,5.7,NaN,NaN,NaN
2,PersonaChat,test,968,0.0,968.0,0.0,NaN,NaN,NaN
3,ConvAI2,train,17878,NaN,NaN,NaN,453.0,17425.0,2.5
4,ConvAI2,validation,1000,NaN,NaN,NaN,57.0,943.0,5.7


In [13]:
# Foco na validação: 1.000 ↔ 1.000
val_convai2 = convai2_dialogs.get("validation", set())
val_pc = pc_dialogs.get("validation", set())
inter_val = val_convai2 & val_pc

print("Validação ConvAI2 (unicas):", len(val_convai2))
print("Validação PersonaChat (unicas):", len(val_pc))
print("Interseção (mesmo diálogo):", len(inter_val))
print("Pct de valid PersonaChat presente em valid ConvAI2:",
      round(len(inter_val) / len(val_pc) * 100, 1) if val_pc else 0.0, "%")

# Sinal complementar: pares de personas (your, partner) como chave
def pair_key(your: Any, partner: Any) -> str:
    return hashlib.sha256(
        (persona_key(your) + "\x00" + persona_key(partner)).encode("utf-8")
    ).hexdigest()

convai2_pairs = set()
for split in convai2:
    data = convai2[split][:]
    for y, p in zip(data["your_persona"], data["partner_persona"]):
        convai2_pairs.add(pair_key(y, p))

pc_pairs = set()
for split in personachat:
    data = personachat[split][:]
    for y, p in zip(data["your_persona_revised"], data["partner_persona_revised"]):
        pc_pairs.add(pair_key(y, p))

print("\n--- Pares de personas (your_revised, partner_revised) ---")
print("ConvAI2 pares unicos:", len(convai2_pairs))
print("PersonaChat pares unicos:", len(pc_pairs))
print("Intersecao:", len(convai2_pairs & pc_pairs),
      f"({round(len(convai2_pairs & pc_pairs) / len(pc_pairs) * 100, 1)}% dos pares PC)")

Validação ConvAI2 (unicas): 1000
Validação PersonaChat (unicas): 1000
Interseção (mesmo diálogo): 57
Pct de valid PersonaChat presente em valid ConvAI2: 5.7 %

--- Pares de personas (your_revised, partner_revised) ---
ConvAI2 pares unicos: 18875
PersonaChat pares unicos: 10895
Intersecao: 1918 (17.6% dos pares PC)


## 7b. Similaridade de diálogos: além da igualdade exata

A igualdade bruta (seção 7) reporta só ~4,7% porque o PersonaChat expande contrações e o ConvAI2 não. Aqui aplicamos a expansão canônica de contrações (`expand_text`) e recomputamos a sobreposição exata, e em seguida usamos uma **similaridade fuzzy** (`difflib.SequenceMatcher`) sobre o texto canônico para quantificar o quão próximas são as conversas, mesmo quando não são idênticas (ex.: ambiguidade `i'd` -> `i would`/`i had`). Por fim, cruzamos o conteúdo do diálogo com o par de personas para revelar onde a *mesma* conversa recebe *diferentes* personas.

In [14]:
# Sobreposição EXATA após expansão de contrações, por split (compara com a bruta).
convai2_dialogs_exp: dict[str, set[str]] = {}
for split in convai2:
    convai2_dialogs_exp[split] = {k for k in (dialog_key_convai2_expanded(t) for t in convai2[split]["turns"]) if k}
pc_dialogs_exp: dict[str, set[str]] = {}
for split in personachat:
    pc_dialogs_exp[split] = {k for k in (dialog_key_personachat_expanded(d) for d in personachat[split]["dialog"]) if k}

convai2_all_exp = set().union(*convai2_dialogs_exp.values())
pc_all_exp = set().union(*pc_dialogs_exp.values())

rows = []
for split, keys in pc_dialogs_exp.items():
    in_any = keys & convai2_all_exp
    raw_pct = round(len(pc_dialogs[split] & convai2_all) / len(pc_dialogs[split]) * 100, 1)
    rows.append({
        "split": split,
        "conversas_pc": len(keys),
        "exata_bruta_pct": raw_pct,
        "exata_expandida_pct": round(len(in_any) / len(keys) * 100, 1) if keys else 0.0,
        "no_convai2_exp": len(in_any),
        "fora_do_convai2": len(keys - convai2_all_exp),
    })
pd.DataFrame(rows)


,split,conversas_pc,exata_bruta_pct,exata_expandida_pct,no_convai2_exp,fora_do_convai2
0,train,8939,5.1,96.1,8592,347
1,validation,1000,5.7,96.3,963,37
2,test,968,0.0,0.0,0,968


In [15]:
# Similaridade fuzzy por conversa do PersonaChat vs melhor candidata no ConvAI2.
# Estratégia de pareamento (rápida, evita O(n^2)):
#   1. se a conversa tem match exato (expandido) no ConvAI2 -> similaridade ~1.0
#   2. senão, se o par de personas (revised, ambas as ordens) existe no ConvAI2 ->
#      similaridade contra essas conversas (resíduo: ambiguidade i'd etc.)
#   3. senão -> sem contraparte (ex.: split de teste)
cv_exp_index: dict[str, list[Any]] = {}
for split in convai2:
    for t in convai2[split]["turns"]:
        k = dialog_key_convai2_expanded(t)
        if k:
            cv_exp_index.setdefault(k, []).append(t)

# índice par-de-personas -> conversas do ConvAI2 (ambas as ordens)
cv_pair_index: dict[str, list[Any]] = {}
for split in convai2:
    d = convai2[split][:]
    for y, p, t in zip(d["your_persona"], d["partner_persona"], d["turns"]):
        ky, kp = persona_key(y), persona_key(p)
        cv_pair_index.setdefault(ky + kp, []).append(t)
        cv_pair_index.setdefault(kp + ky, []).append(t)

buckets = {"exata_expandida": 0, ">=0.99": 0, ">=0.95": 0, ">=0.9": 0, ">=0.8": 0, "<0.8": 0, "sem_contraparte": 0}
total_pc = 0
for split in personachat:
    d = personachat[split][:]
    for y, p, dlg in zip(d["your_persona_revised"], d["partner_persona_revised"], d["dialog"]):
        total_pc += 1
        k = dialog_key_personachat_expanded(dlg)
        if k in cv_exp_index:
            buckets["exata_expandida"] += 1
            continue
        cands = cv_pair_index.get(persona_key(y) + persona_key(p), [])
        if not cands:
            buckets["sem_contraparte"] += 1
            continue
        best = max(conv_similarity(dlg, t) for t in cands)
        if best >= 0.99:
            buckets[">=0.99"] += 1
        elif best >= 0.95:
            buckets[">=0.95"] += 1
        elif best >= 0.9:
            buckets[">=0.9"] += 1
        elif best >= 0.8:
            buckets[">=0.8"] += 1
        else:
            buckets["<0.8"] += 1

sim_df = pd.DataFrame(
    [{"categoria": cat, "conversas": cnt, "pct": round(cnt / total_pc * 100, 1)} for cat, cnt in buckets.items()]
)
display(sim_df)
print(f"Total de conversas do PersonaChat: {total_pc}")
print(f"Praticamente idênticas ao ConvAI2 (exata expandida + >=0.95): "
      f"{buckets['exata_expandida'] + buckets['>=0.99'] + buckets['>=0.95']} "
      f"({round((buckets['exata_expandida'] + buckets['>=0.99'] + buckets['>=0.95']) / total_pc * 100, 1)}%)")


,categoria,conversas,pct
0,exata_expandida,9555,87.6
1,>=0.99,62,0.6
2,>=0.95,2,0.0
3,>=0.9,1,0.0
4,>=0.8,1,0.0
5,<0.8,0,0.0
6,sem_contraparte,1286,11.8


Total de conversas do PersonaChat: 10907
Praticamente idênticas ao ConvAI2 (exata expandida + >=0.95): 9619 (88.2%)


In [16]:
# Para conversas com conteúdo IDÊNTICO (expandido), a persona é a mesma?
# Revela que o PersonaChat anota personas INDEPENDENTES do ConvAI2: a mesma
# conversa aparece com par de personas diferente na maioria dos casos.
cv_exp_to_personas: dict[str, list[tuple[str, str]]] = {}
for split in convai2:
    d = convai2[split][:]
    for y, p, t in zip(d["your_persona"], d["partner_persona"], d["turns"]):
        k = dialog_key_convai2_expanded(t)
        if k:
            cv_exp_to_personas.setdefault(k, []).append((persona_key(y), persona_key(p)))

same_persona_pair = 0
swapped_persona_pair = 0
different_persona_pair = 0
shared_dialog = 0
for split in personachat:
    d = personachat[split][:]
    for y, p, dlg in zip(d["your_persona_revised"], d["partner_persona_revised"], d["dialog"]):
        k = dialog_key_personachat_expanded(dlg)
        if not k or k not in cv_exp_to_personas:
            continue
        shared_dialog += 1
        py, pp = persona_key(y), persona_key(p)
        cvpairs = cv_exp_to_personas[k]
        if (py, pp) in cvpairs:
            same_persona_pair += 1
        elif (pp, py) in cvpairs:
            swapped_persona_pair += 1
        else:
            different_persona_pair += 1

print(f"Conversas do PersonaChat com diálogo idêntico (expandido) no ConvAI2: {shared_dialog}")
print(f"  -> mesmo par de personas (your,partner): {same_persona_pair}")
print(f"  -> par de personas trocado (partner,your): {swapped_persona_pair}")
print(f"  -> par de personas DIFERENTE: {different_persona_pair}")
print()
if different_persona_pair > same_persona_pair + swapped_persona_pair:
    print("Conclusão: mesmo onde o diálogo é idêntico, predomina um par de personas DIFERENTE.")
    print("As anotações de persona são INDEPENDENTES: o PersonaChat reanotou as mesmas")
    print("conversas com sua própria revisão de personas (além de originais + enhanced).")
else:
    print("Conclusão: onde o diálogo é idêntico, o par de personas costuma ser o mesmo.")


Conversas do PersonaChat com diálogo idêntico (expandido) no ConvAI2: 9555
  -> mesmo par de personas (your,partner): 1852
  -> par de personas trocado (partner,your): 0
  -> par de personas DIFERENTE: 7703

Conclusão: mesmo onde o diálogo é idêntico, predomina um par de personas DIFERENTE.
As anotações de persona são INDEPENDENTES: o PersonaChat reanotou as mesmas
conversas com sua própria revisão de personas (além de originais + enhanced).


## 8. Análise de complementaridade e conclusão

Síntese das respostas às perguntas do usuário, com exemplos concretos de conversas compartilhadas e exclusivas.

In [17]:
# Constrói índices pela chave EXATA EXPANDIDA para encontrar exemplos (a bruta
# subestima: só 510 matches brutos vs ~9,5k expandidos).
pc_index: dict[str, tuple[str, int]] = {}
for split in personachat:
    for i, d in enumerate(personachat[split]["dialog"]):
        k = dialog_key_personachat_expanded(d)
        if k and k not in pc_index:
            pc_index[k] = (split, i)

convai2_index: dict[str, tuple[str, int]] = {}
for split in convai2:
    for i, t in enumerate(convai2[split]["turns"]):
        k = dialog_key_convai2_expanded(t)
        if k and k not in convai2_index:
            convai2_index[k] = (split, i)

shared_key = pc_all_exp & convai2_all_exp
print("Conversas compartilhadas (diálogo idêntico expandido):", len(shared_key))

def show_convai2(split: str, idx: int, n_turns: int = 2) -> None:
    row = convai2[split][idx]
    print(f"  [ConvAI2 {split}:{idx}]")
    print(f"  your_persona:      {row['your_persona']}")
    print(f"  partner_persona:   {row['partner_persona']}")
    for t in row["turns"][:n_turns]:
        print(f"  turn {t['turn_id']}: {t['text']!r} -> {t['response']!r}")

def show_pc(split: str, idx: int, n_turns: int = 2) -> None:
    row = personachat[split][idx]
    print(f"  [PersonaChat {split}:{idx}]  dialog_id={row['dialog_id']}")
    print(f"  your_revised:      {row['your_persona_revised']}")
    print(f"  partner_revised:   {row['partner_persona_revised']}")
    dialog = row["dialog"]
    if isinstance(dialog, str):
        dialog = json.loads(dialog)
    for t in dialog[:n_turns]:
        print(f"  turn {t.get('turn')}: {t.get('text')!r} -> {t.get('response')!r}")

print("\n=== Exemplo 1: conversa presente em ambos (mesmo diálogo, contractions) ===")
if shared_key:
    k = next(iter(shared_key))
    cs, ci = convai2_index[k]
    ps, pi = pc_index[k]
    show_convai2(cs, ci)
    show_pc(ps, pi)
else:
    print("(nenhuma conversa compartilhada)")

print("\n=== Exemplo 2: conversa só no ConvAI2 ===")
only_convai2 = convai2_all_exp - pc_all_exp
if only_convai2:
    k = next(iter(only_convai2))
    cs, ci = convai2_index[k]
    show_convai2(cs, ci)
else:
    print("(todas as conversas do ConvAI2 também estão no PersonaChat)")

print("\n=== Exemplo 3: conversa só no PersonaChat (ex.: split de teste) ===")
only_pc = pc_all_exp - convai2_all_exp
if only_pc:
    k = next(iter(only_pc))
    ps, pi = pc_index[k]
    show_pc(ps, pi)
else:
    print("(todas as conversas do PersonaChat também estão no ConvAI2)")


Conversas compartilhadas (diálogo idêntico expandido): 9555

=== Exemplo 1: conversa presente em ambos (mesmo diálogo, contractions) ===
  [ConvAI2 train:1425]
  your_persona:      ["i've a crotch rocket.", "i'm employed.", 'bow and arrows is something i could do all day.', 'i should follow a stricter diet.']
  partner_persona:   ['i have went to school for dance.', 'i enjoy dance.', 'i took ballet lessons when i was young.', 'my clan backs up.', "i've a 401k."]
  turn 0: 'hi , how are you this evening ?' -> 'i am good tired from working today ! finally got a job this week .'
  turn 1: 'whats your job ? i am a ballerina . have been my whole life .' -> 'nice . i clean gutters out . i have to cut back at the archery center now though .'
  [PersonaChat train:1425]  dialog_id=train_001426
  your_revised:      ["i have a crotch rocket.","i am employed.","bow and arrows is something i could do all day.","i should follow a stricter diet."]
  partner_revised:   ["i have went to school for danc

In [18]:
# Confirmação adicional do achado da seção 7b usando a chave EXATA bruta:
# mesmo para pares de personas compartilhados, a igualdade bruta reportava
# "diálogo diferente" — mas isso era APENAS a expansão de contrações.
# Recontamos agora com a chave expandida.
convai2_pair_to_dialogs_exp: dict[str, set[str]] = {}
for split in convai2:
    d = convai2[split][:]
    for y, p, t in zip(d["your_persona"], d["partner_persona"], d["turns"]):
        pk = pair_key(y, p)
        k = dialog_key_convai2_expanded(t)
        if k:
            convai2_pair_to_dialogs_exp.setdefault(pk, set()).add(k)

pc_pair_to_dialogs_exp: dict[str, set[str]] = {}
for split in personachat:
    d = personachat[split][:]
    for y, p, dlg in zip(d["your_persona_revised"], d["partner_persona_revised"], d["dialog"]):
        pk = pair_key(y, p)
        k = dialog_key_personachat_expanded(dlg)
        if k:
            pc_pair_to_dialogs_exp.setdefault(pk, set()).add(k)

shared_pairs = set(convai2_pair_to_dialogs_exp) & set(pc_pair_to_dialogs_exp)
same_pair_same_dialog = sum(1 for pk in shared_pairs if convai2_pair_to_dialogs_exp[pk] & pc_pair_to_dialogs_exp[pk])
same_pair_diff_dialog = len(shared_pairs) - same_pair_same_dialog

print(f"Pares de personas compartilhados (your_revised, partner_revised): {len(shared_pairs)}")
print(f"  -> mesmo par E mesmo diálogo (expandido): {same_pair_same_dialog}")
print(f"  -> mesmo par MAS diálogo diferente (expandido): {same_pair_diff_dialog}")
print()
print("Correção do diagnóstico: com a expansão de contrações, a grande maioria dos")
print("pares compartilhados tem o MESMO diálogo. A impressão anterior de 'diálogo")
print("diferente / versão original' era um artefato da normalização de contrações:")
print("os diálogos são o mesmo conteúdo, só expresso em forma expandida (PersonaChat)")
print("vs. contraída (ConvAI2).")


Pares de personas compartilhados (your_revised, partner_revised): 1918
  -> mesmo par E mesmo diálogo (expandido): 1851
  -> mesmo par MAS diálogo diferente (expandido): 67

Correção do diagnóstico: com a expansão de contrações, a grande maioria dos
pares compartilhados tem o MESMO diálogo. A impressão anterior de 'diálogo
diferente / versão original' era um artefato da normalização de contrações:
os diálogos são o mesmo conteúdo, só expresso em forma expandida (PersonaChat)
vs. contraída (ConvAI2).


In [19]:
# Síntese final
rev_row = overlap_table(convai2_personas, pc_revised, "ConvAI2", "PC revised")
orig_row = overlap_table(convai2_personas, pc_original, "ConvAI2", "PC original")
pc_in_convai2_exp = len(pc_all_exp & convai2_all_exp)
pc_total = len(pc_all_exp)
near_identical = buckets["exata_expandida"] + buckets[">=0.99"] + buckets[">=0.95"]
no_counterpart = buckets["sem_contraparte"] + buckets["<0.8"]
convai2_extra = len(convai2_all_exp - pc_all_exp)

print("=" * 74)
print("RESUMO DA ANÁLISE COMPARATIVA")
print("=" * 74)
print()
print("1) TAMANHO")
print(f"   ConvAI2:     {sum(len(convai2[s]) for s in convai2)} conversas "
      f"(train {len(convai2['train'])} / valid {len(convai2['validation'])})")
print(f"   PersonaChat: {sum(len(personachat[s]) for s in personachat)} conversas "
      f"(train {len(personachat['train'])} / valid {len(personachat['validation'])} / test {len(personachat['test'])})")
print()
print("2) DIÁLOGOS — sobreposição REAL (após expansão de contrações)")
print(f"   Igualdade bruta parecia só {round(len(pc_all & convai2_all)/len(pc_all)*100,1)}%, mas era")
print(f"   artefato de normalização (`i'm` vs `i am`). Após expandir contrações:")
for r in rows:
    print(f"     {r['split']:10s}: {r['exata_expandida_pct']}% das conversas do PersonaChat estão no ConvAI2")
print(f"   Conversas do PersonaChat praticamente idênticas ao ConvAI2 (exata + >=0.95):")
print(f"     {near_identical} / {pc_total} ({round(near_identical/pc_total*100,1)}%)")
print(f"   Sem contraparte no ConvAI2: {no_counterpart} (quase todo o split de teste, 968, + ~318 de treino/valid).")
print(f"   ConvAI2 tem {convai2_extra} conversas de treino sem contraparte no PersonaChat.")
print()
print("3) PERSONAS — anotações INDEPENDENTES, não herdadas")
print(f"   ConvAI2 ∩ PC_revised  = {rev_row['intersecao']}  (Jaccard {rev_row['jaccard']})")
print(f"   ConvAI2 ∩ PC_original = {orig_row['intersecao']}  (Jaccard {orig_row['jaccard']})")
print(f"   ConvAI2 corresponde à versão REVISED, mas só {rev_row['intersecao']} de {rev_row['tamanho_b']}")
print(f"   ({round(rev_row['intersecao']/rev_row['tamanho_b']*100,1)}%) das personas revised do PC estão no ConvAI2.")
print(f"   E nas {shared_dialog} conversas com diálogo IDÊNTICO, o par de personas difere em")
print(f"   {different_persona_pair} casos (mesmo em só {same_persona_pair + swapped_persona_pair}).")
print(f"   => O PersonaChat reanotou as mesmas conversas com SUA própria revisão de personas.")
print()
print("4) O QUE PERSONACHAT ADICIONA sobre o ConvAI2:")
print("   - Personas ORIGINAIS (não-revisadas) — zero overlap com ConvAI2")
print("   - 4 colunas de personas ENHANCED (descrições visuais ricas)")
print("   - Coluna `candidates` por turn (respostas candidatas ranqueadas)")
print("   - Split de TESTE (968 conversas) ausente no ConvAI2")
print("   - `dialog_id` canônico (ex.: train_000001)")
print()
print("5) O QUE CONVAI2 ADICIONA sobre o PersonaChat:")
print(f"   - {convai2_extra} conversas extras de treino (corpus bem maior: 17.878 vs 8.939)")
print("   - Formato 'one row' com turns estruturados (turn_id/text/response), sem candidates")
print()
print("6) SÃO COMPLEMENTARES? SIM, com nuance:")
print("   - DIÁLOGOS: quase idênticos onde se sobrepõem (~88% do PC, train↔train e")
print("     valid↔valid), diferindo só por expansão de contrações. PC adiciona o split")
print("     de teste; ConvAI2 adiciona ~9,3k conversas de treino.")
print("   - PERSONAS: independentes. Compartilham linhagem 'Both Revised' mas o PC")
print("     reanotou com suas próprias revisões + originais + enhanced.")
print("   => Mesmo corpus de diálogos (parcial), anotações de persona distintas.")
print("      Use ConvAI2 p/ volume de treino e diálogos limpos; PersonaChat p/ personas")
print("      ricas (original/revised/enhanced), candidates e split de teste.")


RESUMO DA ANÁLISE COMPARATIVA

1) TAMANHO
   ConvAI2:     18878 conversas (train 17878 / valid 1000)
   PersonaChat: 10907 conversas (train 8939 / valid 1000 / test 968)

2) DIÁLOGOS — sobreposição REAL (após expansão de contrações)
   Igualdade bruta parecia só 4.7%, mas era
   artefato de normalização (`i'm` vs `i am`). Após expandir contrações:
     train     : 96.1% das conversas do PersonaChat estão no ConvAI2
     validation: 96.3% das conversas do PersonaChat estão no ConvAI2
     test      : 0.0% das conversas do PersonaChat estão no ConvAI2
   Conversas do PersonaChat praticamente idênticas ao ConvAI2 (exata + >=0.95):
     9619 / 10907 (88.2%)
   Sem contraparte no ConvAI2: 1286 (quase todo o split de teste, 968, + ~318 de treino/valid).
   ConvAI2 tem 9323 conversas de treino sem contraparte no PersonaChat.

3) PERSONAS — anotações INDEPENDENTES, não herdadas
   ConvAI2 ∩ PC_revised  = 3787  (Jaccard 0.2983)
   ConvAI2 ∩ PC_original = 0  (Jaccard 0.0)
   ConvAI2 corresponde 